In [ ]:
from __future__ import annotations

import os
import sys
import csv
import json
import time
import random
from math import cos, pi
from datetime import datetime
from collections import Counter
from typing import List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Project root setup
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from configs.config import Config
from models.vit_temporal import ViTS_TemporalTransformer
from utils.device import empty_cache

# Reuse the FaceForensics++ helpers from your evaluation script
import test_faceforensics as ffpp

In [2]:
# ============================================================
# USER CONFIG
# ============================================================

RUN_MODE = "train"  
# allowed: "prepare_split_only", "train", "test_only"

FFPP_ROOT = os.path.join(PROJECT_ROOT, "FaceForensics++_C23")
METADATA_CSV = os.path.join(FFPP_ROOT, "csv", "FF++_Metadata.csv")

MODEL_NAME = "FFpp_C23"

# Cumulative epoch target:
# Example:
#   day 1 -> TOTAL_EPOCHS = 4
#   day 2 -> TOTAL_EPOCHS = 7
#   day 3 -> TOTAL_EPOCHS = 10
TOTAL_EPOCHS = 4

BATCH_SIZE_OVERRIDE = 0   # 0 = use Config.BATCH_SIZE
SEED = 42

VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

CATEGORIES = ""          # e.g. "original,Deepfakes" or "" for all
MAX_VIDEOS = 0           # 0 = all
SKIP_MISSING = True

NO_PRETRAINED = False
NO_RESUME = False

COOLDOWN_SECONDS = 0     # set >0 if you want pause between epochs
MAKE_PLOTS = True

DEVICE_OVERRIDE = ""     # "", "cuda", "cpu", "mps"

# split CSV folder under checkpoints/<MODEL_NAME>/splits/
# these files are persistent and reused across different days

In [3]:
def apply_config_model_name(model_name: str) -> str:
    Config.MODEL_NAME = model_name
    Config.CHECKPOINT_DIR = os.path.join(Config.PROJECT_ROOT, "checkpoints", model_name)
    return Config.CHECKPOINT_DIR

CHECKPOINT_DIR = apply_config_model_name(MODEL_NAME)
SPLIT_DIR = os.path.join(CHECKPOINT_DIR, "splits")

TRAIN_CSV = os.path.join(SPLIT_DIR, "train_split.csv")
VAL_CSV = os.path.join(SPLIT_DIR, "val_split.csv")
TEST_CSV = os.path.join(SPLIT_DIR, "test_split.csv")
SPLIT_META_JSON = os.path.join(SPLIT_DIR, "split_meta.json")

RESUME_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")

DEVICE = torch.device(DEVICE_OVERRIDE) if DEVICE_OVERRIDE else Config.DEVICE

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

print("Checkpoint dir:", CHECKPOINT_DIR)
print("Split dir     :", SPLIT_DIR)
print("Device        :", DEVICE)

Checkpoint dir: /Users/krrishkumar/Documents/Personal Learning/BTP/Multi-Modal-DeepFake-Detection-using-CNN-RNN-and-Rppg/checkpoints/FFpp_C23
Split dir     : /Users/krrishkumar/Documents/Personal Learning/BTP/Multi-Modal-DeepFake-Detection-using-CNN-RNN-and-Rppg/checkpoints/FFpp_C23/splits
Device        : mps


In [4]:
class FaceForensicsClipDataset(Dataset):
    """One clip per video; paths must exist and be readable."""

    def __init__(
        self,
        video_paths: Sequence[str],
        labels: Sequence[int],
        clip_len: int,
        transform: transforms.Compose,
    ):
        if len(video_paths) != len(labels):
            raise ValueError("video_paths and labels length mismatch")
        self.paths = list(video_paths)
        self.labels = list(labels)
        self.clip_len = clip_len
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        path = self.paths[idx]
        frames = ffpp.read_frames_rgb(path)
        if not frames:
            raise ValueError(f"No decodable frames in video: {path}")
        clip = ffpp.frames_to_clip(frames, self.clip_len, self.transform)
        return clip, self.labels[idx]

In [5]:
class AverageMeter:
    def __init__(self) -> None:
        self.reset()

    def reset(self) -> None:
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1) -> None:
        self.val = float(val)
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(1, self.count)


def calculate_accuracy(outputs: torch.Tensor, targets: torch.Tensor) -> float:
    batch_size = targets.size(0)
    _, pred = outputs.topk(1, 1, True)
    pred = pred.t()
    correct = pred.eq(targets.view(1, -1))
    return 100.0 * correct.float().sum().item() / batch_size


def train_epoch(
    epoch: int,
    total_epochs: int,
    data_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    losses = AverageMeter()
    accuracies = AverageMeter()

    pbar = tqdm(
        data_loader,
        desc=f"Train {epoch}/{total_epochs}",
        bar_format="{l_bar}{bar:30}{r_bar}",
        leave=True,
    )

    for inputs, targets in pbar:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        losses.update(loss.item(), inputs.size(0))
        accuracies.update(calculate_accuracy(outputs, targets), inputs.size(0))
        pbar.set_postfix_str(f"loss={losses.avg:.4f} acc={accuracies.avg:.2f}%")

    return losses.avg, accuracies.avg


def validate(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    desc: str = "Validate",
) -> Tuple[List[int], List[int], np.ndarray, float, float]:
    model.eval()

    losses = AverageMeter()
    accuracies = AverageMeter()

    true_labels: List[int] = []
    predictions: List[int] = []
    all_probs: List[np.ndarray] = []

    pbar = tqdm(
        data_loader,
        desc=desc,
        bar_format="{l_bar}{bar:30}{r_bar}",
        leave=True,
    )

    with torch.no_grad():
        for inputs, targets in pbar:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, dtype=torch.long, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            probs = torch.softmax(outputs, dim=1)

            true_labels.extend(targets.cpu().numpy().tolist())
            predictions.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy())

            losses.update(loss.item(), inputs.size(0))
            accuracies.update(calculate_accuracy(outputs, targets), inputs.size(0))
            pbar.set_postfix_str(f"loss={losses.avg:.4f} acc={accuracies.avg:.2f}%")

    return true_labels, predictions, np.array(all_probs), losses.avg, accuracies.avg

    def save_checkpoint(
    checkpoint_dir: str,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    train_loss_avg: List[float],
    train_accuracy: List[float],
    val_loss_avg: List[float],
    val_accuracy: List[float],
    val_true: Optional[List[int]] = None,
    val_preds: Optional[List[int]] = None,
    val_probs: Optional[np.ndarray] = None,
    best_val_acc: float = 0.0,
    filename: str = "checkpoint.pt",
    ) -> str:
        os.makedirs(checkpoint_dir, exist_ok=True)
        path = os.path.join(checkpoint_dir, filename)
    
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "train_loss_avg": train_loss_avg,
                "train_accuracy": train_accuracy,
                "val_loss_avg": val_loss_avg,
                "val_accuracy": val_accuracy,
                "val_true_labels": val_true,
                "val_predictions": val_preds,
                "val_probabilities": val_probs,
                "best_val_acc": best_val_acc,
            },
            path,
        )
        return path


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    device: torch.device,
):
    ckpt = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    return {
        "epoch": ckpt["epoch"],
        "train_loss_avg": ckpt.get("train_loss_avg", []),
        "train_accuracy": ckpt.get("train_accuracy", []),
        "val_loss_avg": ckpt.get("val_loss_avg", []),
        "val_accuracy": ckpt.get("val_accuracy", []),
        "val_true_labels": ckpt.get("val_true_labels"),
        "val_predictions": ckpt.get("val_predictions"),
        "val_probabilities": ckpt.get("val_probabilities"),
        "best_val_acc": ckpt.get("best_val_acc", 0.0),
    }


def cooldown(seconds: int) -> None:
    if seconds <= 0:
        return
    print(f"\nCooldown: waiting {seconds}s")
    for remaining in range(seconds, 0, -1):
        mins, secs = divmod(remaining, 60)
        print(f"\r  {mins:02d}:{secs:02d} remaining", end="", flush=True)
        time.sleep(1)
    print("\r  done                    ")

In [6]:
def norm_abs(path: str) -> str:
    return os.path.normpath(os.path.abspath(path))


def to_rel_if_under_root(path: str, root: str) -> str:
    path = os.path.normpath(path)
    root = os.path.normpath(root)
    if path.startswith(root + os.sep):
        return os.path.relpath(path, root)
    return path


def save_split_csv(csv_path: str, ffpp_root: str, paths: Sequence[str], labels: Sequence[int]) -> None:
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["File Path", "Label"])
        for p, y in zip(paths, labels):
            rel = to_rel_if_under_root(os.path.normpath(p), os.path.normpath(ffpp_root))
            label_str = "FAKE" if y == 0 else "REAL"
            writer.writerow([rel, label_str])


def load_split_csv(csv_path: str, ffpp_root: str) -> Tuple[List[str], List[int]]:
    paths, labels = [], []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            p = row["File Path"].strip()
            lab = row["Label"].strip().upper()

            full = p if os.path.isabs(p) else os.path.join(ffpp_root, p)
            full = os.path.normpath(full)

            if not os.path.isfile(full):
                raise FileNotFoundError(f"Missing file listed in split CSV: {full}")

            if lab not in {"FAKE", "REAL"}:
                raise ValueError(f"Invalid label '{lab}' in {csv_path}")

            y = 0 if lab == "FAKE" else 1
            paths.append(full)
            labels.append(y)

    return paths, labels


def save_split_metadata(
    json_path: str,
    seed: int,
    val_fraction: float,
    test_fraction: float,
    categories: str,
    max_videos: int,
    train_size: int,
    val_size: int,
    test_size: int,
):
    meta = {
        "seed": seed,
        "val_fraction": val_fraction,
        "test_fraction": test_fraction,
        "categories": categories,
        "max_videos": max_videos,
        "train_size": train_size,
        "val_size": val_size,
        "test_size": test_size,
        "created_at": datetime.now().isoformat(),
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

In [7]:
def collect_samples(
    ffpp_root: str,
    metadata_csv: str,
    categories_filter: Optional[set],
    skip_missing: bool,
    max_total: Optional[int],
    seed: int,
) -> Tuple[List[str], List[int]]:
    rows = ffpp.load_metadata_rows(metadata_csv)

    if categories_filter is not None:
        rows = [r for r in rows if r["category"] in categories_filter]

    paths: List[str] = []
    labels: List[int] = []
    skipped_missing = 0

    for r in rows:
        full = os.path.join(ffpp_root, r["rel_path"])
        full = os.path.normpath(full)

        if not os.path.isfile(full):
            if skip_missing:
                skipped_missing += 1
                continue
            raise FileNotFoundError(f"Video missing: {full}")

        label = 0 if r["label"].upper() == "FAKE" else 1
        paths.append(full)
        labels.append(label)

    if skipped_missing:
        print(f"[Data] Skipped {skipped_missing} missing file(s).")

    if len(set(labels)) < 2:
        raise RuntimeError("Need both FAKE and REAL samples after filtering.")

    if max_total is not None and max_total > 0 and len(paths) > max_total:
        rng = random.Random(seed)
        idx = list(range(len(paths)))
        rng.shuffle(idx)
        idx = idx[:max_total]
        paths = [paths[i] for i in idx]
        labels = [labels[i] for i in idx]
        print(f"[Data] Subsampled to {max_total} videos.")

    return paths, labels


def stratified_train_val_test_split(
    paths: List[str],
    labels: List[int],
    val_fraction: float,
    test_fraction: float,
    seed: int,
) -> Tuple[List[str], List[str], List[str], List[int], List[int], List[int]]:
    if not (0 < val_fraction < 1 and 0 < test_fraction < 1):
        raise ValueError("val_fraction and test_fraction must be in (0,1)")
    if val_fraction + test_fraction >= 1.0:
        raise ValueError("val_fraction + test_fraction must be < 1")

    train_val_p, test_p, train_val_y, test_y = train_test_split(
        paths,
        labels,
        test_size=test_fraction,
        random_state=seed,
        stratify=labels,
    )

    val_ratio_of_remainder = val_fraction / (1.0 - test_fraction)

    train_p, val_p, train_y, val_y = train_test_split(
        train_val_p,
        train_val_y,
        test_size=val_ratio_of_remainder,
        random_state=seed,
        stratify=train_val_y,
    )

    return train_p, val_p, test_p, train_y, val_y, test_y


def prepare_or_load_fixed_splits(
    ffpp_root: str,
    metadata_csv: str,
    categories: str,
    skip_missing: bool,
    max_videos: int,
    seed: int,
    val_fraction: float,
    test_fraction: float,
    train_csv: str,
    val_csv: str,
    test_csv: str,
    split_meta_json: str,
):
    split_files_exist = all(os.path.isfile(p) for p in [train_csv, val_csv, test_csv])

    if split_files_exist:
        print("[Split] Loading existing fixed split CSVs...")
        train_p, train_y = load_split_csv(train_csv, ffpp_root)
        val_p, val_y = load_split_csv(val_csv, ffpp_root)
        test_p, test_y = load_split_csv(test_csv, ffpp_root)
        return train_p, val_p, test_p, train_y, val_y, test_y

    print("[Split] No split CSVs found. Creating fixed train/val/test split now...")

    cats_filter = None
    if categories.strip():
        cats_filter = {c.strip() for c in categories.split(",") if c.strip()}

    max_v = max_videos if max_videos > 0 else None

    paths, labels = collect_samples(
        ffpp_root=ffpp_root,
        metadata_csv=metadata_csv,
        categories_filter=cats_filter,
        skip_missing=skip_missing,
        max_total=max_v,
        seed=seed,
    )

    train_p, val_p, test_p, train_y, val_y, test_y = stratified_train_val_test_split(
        paths=paths,
        labels=labels,
        val_fraction=val_fraction,
        test_fraction=test_fraction,
        seed=seed,
    )

    save_split_csv(train_csv, ffpp_root, train_p, train_y)
    save_split_csv(val_csv, ffpp_root, val_p, val_y)
    save_split_csv(test_csv, ffpp_root, test_p, test_y)

    save_split_metadata(
        split_meta_json,
        seed=seed,
        val_fraction=val_fraction,
        test_fraction=test_fraction,
        categories=categories,
        max_videos=max_videos,
        train_size=len(train_p),
        val_size=len(val_p),
        test_size=len(test_p),
    )

    print("[Split] Saved fixed split CSVs:")
    print("   ", train_csv)
    print("   ", val_csv)
    print("   ", test_csv)

    return train_p, val_p, test_p, train_y, val_y, test_y

In [8]:
assert os.path.isfile(METADATA_CSV), f"Metadata CSV not found: {METADATA_CSV}"
assert os.path.isdir(FFPP_ROOT), f"FF++ root not found: {FFPP_ROOT}"

train_p, val_p, test_p, train_y, val_y, test_y = prepare_or_load_fixed_splits(
    ffpp_root=norm_abs(FFPP_ROOT),
    metadata_csv=norm_abs(METADATA_CSV),
    categories=CATEGORIES,
    skip_missing=SKIP_MISSING,
    max_videos=MAX_VIDEOS,
    seed=SEED,
    val_fraction=VAL_FRACTION,
    test_fraction=TEST_FRACTION,
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    test_csv=TEST_CSV,
    split_meta_json=SPLIT_META_JSON,
)

def fake_real_counts(ys: Sequence[int]) -> Tuple[int, int]:
    c = Counter(ys)
    return c[0], c[1]

tfk, trk = fake_real_counts(train_y)
vfk, vrk = fake_real_counts(val_y)
xfk, xrk = fake_real_counts(test_y)

print("\nFixed split summary")
print("-------------------")
print(f"Train: {len(train_p)}  (FAKE={tfk}, REAL={trk})")
print(f"Val  : {len(val_p)}  (FAKE={vfk}, REAL={vrk})")
print(f"Test : {len(test_p)}  (FAKE={xfk}, REAL={xrk})")

[Split] Loading existing fixed split CSVs...

Fixed split summary
-------------------
Train: 4900  (FAKE=4200, REAL=700)
Val  : 1050  (FAKE=900, REAL=150)
Test : 1050  (FAKE=900, REAL=150)


In [9]:
if RUN_MODE == "prepare_split_only":
    print("Fixed split CSVs are ready.")
    print(TRAIN_CSV)
    print(VAL_CSV)
    print(TEST_CSV)

In [10]:
if RUN_MODE in {"train", "test_only"}:
    tfm = ffpp.build_eval_transform(Config.IM_SIZE, Config.MEAN, Config.STD)

    train_ds = FaceForensicsClipDataset(train_p, train_y, Config.CLIP_LEN, tfm)
    val_ds = FaceForensicsClipDataset(val_p, val_y, Config.CLIP_LEN, tfm)
    test_ds = FaceForensicsClipDataset(test_p, test_y, Config.CLIP_LEN, tfm)

    batch_size = BATCH_SIZE_OVERRIDE if BATCH_SIZE_OVERRIDE > 0 else Config.BATCH_SIZE
    pin_memory = (DEVICE.type == "cuda")

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=pin_memory,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=pin_memory,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=pin_memory,
    )

    print("Batch size:", batch_size)
    print("Train batches:", len(train_loader))
    print("Val batches  :", len(val_loader))
    print("Test batches :", len(test_loader))

Batch size: 8
Train batches: 613
Val batches  : 132
Test batches : 132


In [11]:
if RUN_MODE in {"train", "test_only"}:
    model = ViTS_TemporalTransformer(
        num_classes=Config.NUM_CLASSES,
        pretrained=not NO_PRETRAINED,
        temporal_layers=Config.TEMPORAL_LAYERS,
        temporal_heads=Config.TEMPORAL_HEADS,
        temporal_ff=Config.TEMPORAL_FF,
        temporal_dropout=Config.TEMPORAL_DROPOUT,
        t_max=Config.T_MAX,
        use_cls_token=Config.USE_CLS_TOKEN,
    ).to(DEVICE)

    vit_params = model.frame_encoder.parameters()
    temp_head_params = list(model.temporal.parameters()) + list(model.head.parameters())

    optimizer = torch.optim.AdamW(
        [
            {"params": vit_params, "lr": 3e-5},
            {"params": temp_head_params, "lr": 1e-4},
        ],
        weight_decay=Config.WEIGHT_DECAY,
    )

    total_steps = max(1, TOTAL_EPOCHS * max(1, len(train_loader)))
    warmup_steps = int(0.1 * total_steps)

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + cos(pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    criterion = nn.CrossEntropyLoss().to(DEVICE)

    train_loss_avg: List[float] = []
    train_accuracy: List[float] = []
    val_loss_avg: List[float] = []
    val_accuracy: List[float] = []
    best_val_acc = 0.0
    start_epoch = 1

    Config.print_config()

  Configuration
  Platform         : Darwin arm64
  Device           : mps
  Backbone         : vit_small
  Batch size       : 8
  Clip length      : 30
  Num workers      : 0
  Pin memory       : False
  Data dir         : /Users/krrishkumar/Documents/Personal Learning/BTP/Multi-Modal-DeepFake-Detection-using-CNN-RNN-and-Rppg
  Model name       : FFpp_C23
  Checkpoint dir   : /Users/krrishkumar/Documents/Personal Learning/BTP/Multi-Modal-DeepFake-Detection-using-CNN-RNN-and-Rppg/checkpoints/FFpp_C23
  Epochs           : 20
  Learning rate    : 0.0001
  Weight decay     : 0.05
  Image size       : 224
  Num classes      : 2
  Temporal layers  : 4
  Temporal heads   : 6
  CLS token        : True


/Users/krrishkumar/Documents/Personal Learning/BTP/Multi-Modal-DeepFake-Detection-using-CNN-RNN-and-Rppg/venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [12]:
if RUN_MODE == "train":
    if (not NO_RESUME) and os.path.isfile(RESUME_PATH):
        ckpt_info = load_checkpoint(
            path=RESUME_PATH,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            device=DEVICE,
        )
        start_epoch = int(ckpt_info["epoch"]) + 1
        train_loss_avg = list(ckpt_info["train_loss_avg"])
        train_accuracy = list(ckpt_info["train_accuracy"])
        val_loss_avg = list(ckpt_info["val_loss_avg"])
        val_accuracy = list(ckpt_info["val_accuracy"])
        best_val_acc = float(ckpt_info["best_val_acc"])

        if os.path.isfile(BEST_MODEL_PATH):
            best_ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
            best_val_acc = max(best_val_acc, float(best_ckpt.get("best_val_acc", 0.0)))

        print(f"Resumed from epoch {start_epoch - 1}")
        print(f"Best val acc so far: {best_val_acc:.2f}%")
    else:
        print("No checkpoint loaded. Training will start from scratch.")

No checkpoint loaded. Training will start from scratch.


In [ ]:
if RUN_MODE == "train":
    if start_epoch > TOTAL_EPOCHS:
        print(
            f"No training needed: checkpoint is already at epoch {start_epoch - 1}, "
            f"which is >= TOTAL_EPOCHS ({TOTAL_EPOCHS})."
        )
    else:
        print("=" * 70)
        print(f"Training epochs {start_epoch} .. {TOTAL_EPOCHS} on {DEVICE}")
        print("=" * 70)

        last_val_true = None
        last_val_pred = None
        last_val_probs = None

        for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
            t0 = time.time()
            lr_vit = optimizer.param_groups[0]["lr"]
            lr_head = optimizer.param_groups[1]["lr"]

            print("\n" + "─" * 70)
            print(
                f"Epoch {epoch}/{TOTAL_EPOCHS} | "
                f"lr_vit={lr_vit:.2e} | lr_head={lr_head:.2e} | "
                f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
            )
            print("─" * 70)

            tl, ta = train_epoch(
                epoch=epoch,
                total_epochs=TOTAL_EPOCHS,
                data_loader=train_loader,
                model=model,
                criterion=criterion,
                optimizer=optimizer,
                scheduler=scheduler,
                device=DEVICE,
            )
            train_loss_avg.append(tl)
            train_accuracy.append(ta)

            val_true, val_pred, val_probs, vl, va = validate(
                model=model,
                data_loader=val_loader,
                criterion=criterion,
                device=DEVICE,
                desc=f"Val {epoch}/{TOTAL_EPOCHS}",
            )
            val_loss_avg.append(vl)
            val_accuracy.append(va)

            last_val_true = val_true
            last_val_pred = val_pred
            last_val_probs = val_probs

            elapsed = time.time() - t0
            print(f"\nTrain Loss: {tl:.4f} | Train Acc: {ta:.2f}%")
            print(f"Val   Loss: {vl:.4f} | Val   Acc: {va:.2f}%")
            print(f"Time      : {elapsed / 60:.2f} min")

            improved = va > best_val_acc
            if improved:
                best_val_acc = va
                best_path = save_checkpoint(
                    checkpoint_dir=CHECKPOINT_DIR,
                    epoch=epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    train_loss_avg=train_loss_avg,
                    train_accuracy=train_accuracy,
                    val_loss_avg=val_loss_avg,
                    val_accuracy=val_accuracy,
                    val_true=val_true,
                    val_preds=val_pred,
                    val_probs=val_probs,
                    best_val_acc=best_val_acc,
                    filename="best_model.pt",
                )
                print(f"Best model saved -> {best_path} (Val Acc = {best_val_acc:.2f}%)")
            else:
                print(f"Val acc {va:.2f}% did not improve best {best_val_acc:.2f}%")

            ckpt_path = save_checkpoint(
                checkpoint_dir=CHECKPOINT_DIR,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                train_loss_avg=train_loss_avg,
                train_accuracy=train_accuracy,
                val_loss_avg=val_loss_avg,
                val_accuracy=val_accuracy,
                val_true=val_true,
                val_preds=val_pred,
                val_probs=val_probs,
                best_val_acc=best_val_acc,
                filename="checkpoint.pt",
            )
            print(f"Checkpoint saved -> {ckpt_path}")

            empty_cache(DEVICE)

            if epoch < TOTAL_EPOCHS and COOLDOWN_SECONDS > 0:
                cooldown(COOLDOWN_SECONDS)

        print("\n" + "=" * 70)
        print("Training complete")
        print(f"Best Val Acc: {best_val_acc:.2f}%")
        print("=" * 70)

        if last_val_true is not None and last_val_pred is not None:
            print("\nValidation classification report (last epoch run in this session):")
            print(
                classification_report(
                    last_val_true,
                    last_val_pred,
                    labels=[0, 1],
                    target_names=["Fake (0)", "Real (1)"],
                    digits=4,
                )
            )

Training epochs 1 .. 4 on mps

──────────────────────────────────────────────────────────────────────
Epoch 1/4 | lr_vit=1.22e-07 | lr_head=4.08e-07 | 2026-04-05 11:55:41
──────────────────────────────────────────────────────────────────────


Train 1/4:   1%|▎                             | 7/613 [01:33<2:24:13, 14.28s/it,

In [ ]:
if RUN_MODE in {"train", "test_only"}:
    if os.path.isfile(BEST_MODEL_PATH):
        best_ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best_ckpt["model_state_dict"])
        print(f"Loaded best model from: {BEST_MODEL_PATH}")
        print(f"Stored best val acc   : {best_ckpt.get('best_val_acc', 'n/a')}")
    elif os.path.isfile(RESUME_PATH):
        last_ckpt = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(last_ckpt["model_state_dict"])
        print("best_model.pt not found, loaded checkpoint.pt instead")
    else:
        raise FileNotFoundError("No best_model.pt or checkpoint.pt found for testing")

    true_te, pred_te, probs_te, loss_te, acc_te = validate(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=DEVICE,
        desc="Held-out Test",
    )

    print("\n" + "=" * 70)
    print("Held-out test set results")
    print("=" * 70)
    print(f"Test loss: {loss_te:.4f}")
    print(f"Test acc : {acc_te:.2f}%")

    # class 0 = FAKE, so probs_te[:, 0] is P(fake)
    p_fake = probs_te[:, 0]
    y_bin = [1 if t == 0 else 0 for t in true_te]

    try:
        auc = roc_auc_score(y_bin, p_fake)
        print(f"ROC-AUC (FAKE positive, P(fake)): {auc:.4f}")
    except ValueError:
        print("ROC-AUC: n/a (single class present in test set)")

    print("\nClassification report (held-out test):")
    print(
        classification_report(
            true_te,
            pred_te,
            labels=[0, 1],
            target_names=["Fake (0)", "Real (1)"],
            digits=4,
        )
    )

In [ ]:
if RUN_MODE == "train" and MAKE_PLOTS:
    if os.path.isfile(RESUME_PATH):
        ckpt = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
        train_loss_plot = ckpt.get("train_loss_avg", [])
        train_acc_plot = ckpt.get("train_accuracy", [])
        val_loss_plot = ckpt.get("val_loss_avg", [])
        val_acc_plot = ckpt.get("val_accuracy", [])
    else:
        train_loss_plot = train_loss_avg
        train_acc_plot = train_accuracy
        val_loss_plot = val_loss_avg
        val_acc_plot = val_accuracy

    def _style_axes(ax):
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("#333333")
            spine.set_linewidth(1.3)
        ax.tick_params(axis="both", colors="#333333", labelsize=11, length=4, width=1.0)
        ax.grid(True, alpha=0.3, linestyle="--", color="gray")
        ax.set_facecolor("white")

    epochs_axis = range(1, len(train_loss_plot) + 1)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs_axis, train_loss_plot, label="Training loss", marker="o", markersize=4)
    ax.plot(epochs_axis, val_loss_plot, label="Validation loss", marker="s", markersize=4)
    ax.set_title("Training and Validation Loss (FaceForensics++)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    _style_axes(ax)
    ax.legend()
    loss_fig_path = os.path.join(CHECKPOINT_DIR, "loss_plot_ffpp.png")
    fig.savefig(loss_fig_path, dpi=200, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs_axis, train_acc_plot, label="Training acc", marker="o", markersize=4)
    ax.plot(epochs_axis, val_acc_plot, label="Validation acc", marker="s", markersize=4)
    ax.set_title("Training and Validation Accuracy (FaceForensics++)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    _style_axes(ax)
    ax.legend()
    acc_fig_path = os.path.join(CHECKPOINT_DIR, "accuracy_plot_ffpp.png")
    fig.savefig(acc_fig_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:")
    print(loss_fig_path)
    print(acc_fig_path)